In [0]:
%sql
create volume if not exists healthcare_lakehouse.bronze._checkpoints

In [0]:
from pyspark.sql import functions as F 


MINIO_ENDPOINT = "http://127.0.0.1:9002"
MINIO_ACCESS_KEY = dbutils.secrets.get(scope="meridian-legacy-db", key="minio-access-key")
MINIO_SECRET_KEY = dbutils.secrets.get(scope="meridian-legacy-db", key="minio-secret-key")

spark.conf.set("fs.s3a.endpoint", MINIO_ENDPOINT)
spark.conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
spark.conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
spark.conf.set("fs.s3a.path.style.access", "true")
spark.conf.set("fs.s3a.connection.ssl.enabled", "false")



BUCKET = "legacy-fileshare"
CATALOG = "healthcare_lakehouse"
SCHEMA = "bronze"
CHECKPOINT_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/_checkpoints/fileshare"
MIGRATION_BATCH_ID = spark.sql("select uuid() as id").collect()[0]['id']


def ingest_small_documents(prefix:str, target_table:str, source_label:str):
    """FHIR/C-CDA path: embed raw bytes, one row per file."""
    src_path= f"s3a://{BUCKET}/{prefix}/"
    checkpoint = f"{CHECKPOINT_ROOT}/{target_table}"

    df= (
        spark.readStream.format("cloudfiles")
        .option("cloudFiles.format", "binaryFile")
        .option("cloudFiles.schemaHints", "content binary")
        .option("cloudFiles.schemaLocation", checkpoint)
        .load(src_path)
        .withColumn("_legacy_source_path", F.input_file_name())
        .withColumn("_legacy_source_system",F.lit(source_label))
        .withColumn("_migration_batch_id", F.lit(MIGRATION_BATCH_ID))
        .withColumn("_migrated_at", F.current_timestamp())
        
    )

    return(
        df.writeStream.format("delta")
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.{SCHEMA}.{target_table}")
    )


def catalog_large_binares(prefix:str, target_table:str, source_lable:str, copy_dest_container:str):
    src_path = f"s3a://{BUCKET}/{prefix}/"
    dest_path = f"abfss://{copy_dest_container}@mhealthcarelakehouse.dfs.core.windows.net/{prefix}/"
    checkpoint = f"{CHECKPOINT_ROOT}/{target_table}"

    dbutils.fs.cp(src_path,dest_path, recurse=True)


    df = (
        spark.readStream.format("cloudfiles")
        .option("cloudFiles.format", "binaryFile")
        .option("cloudFiles.SchemaLocation", checkpoint)
        .option("cloudFiles.schemaHints", "content binary")
        .load(dest_path)
        .drop("content")
        .withColumn("_legacy_source_path", F.input_file_name())
        .withColumn("_legacy_source_system", F.lit(source_lable))
        .withColumn("_migration_batch_id", F.lit(MIGRATION_BATCH_ID))
        .withColumn("_migrated_at", F.current_timestamp())
    )

    return (
        df.writeStream.format("delta")
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.{SCHEMA}.{target_table}")
    )

q_ccda = ingest_small_documents("ccda","ccda_documents", "legacy_fileshare_ccda")
q_ccda.awaitTermination()

q_fhir = ingest_small_documents("fhir","fhir_documents", "legacy_fileshare_fhir")
q_fhir.awaitTermination()

q_images = catalog_large_binares("images","imaging_files_catalog", "legacy_fileshare_images", "bronze")
q_images.awaitTermination()


for tbl in ["ccda_documents", "fhir_bundles", "imaging_files_catalog"]:
    n = spark.table(f"{CATALOG}.{SCHEMA}.{tbl}").count()
    print(f"{tbl}: {n}")

In [0]:
from pyspark.sql import functions as F 


MINIO_ENDPOINT = "http://127.0.0.1:9002"
MINIO_ACCESS_KEY = dbutils.secrets.get(scope="meridian-legacy-db", key="minio-access-key")
MINIO_SECRET_KEY = dbutils.secrets.get(scope="meridian-legacy-db", key="minio-secret-key")

spark.conf.set("fs.s3a.endpoint", MINIO_ENDPOINT)
spark.conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
spark.conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
spark.conf.set("fs.s3a.path.style.access", "true")
spark.conf.set("fs.s3a.connection.ssl.enabled", "false")

In [0]:
import concurrent.futures
from pyspark.sql import functions as F
 
# --- MinIO connectivity (via the Cloudflare-tunneled local proxy port) ---------------------
MINIO_ENDPOINT = "http://127.0.0.1:9002"
MINIO_ACCESS_KEY = dbutils.secrets.get(scope="meridian-legacy-db", key="minio-access-key")
MINIO_SECRET_KEY = dbutils.secrets.get(scope="meridian-legacy-db", key="minio-secret-key")
 
spark.conf.set("fs.s3a.endpoint", MINIO_ENDPOINT)
spark.conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
spark.conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
spark.conf.set("fs.s3a.path.style.access", "true")
spark.conf.set("fs.s3a.connection.ssl.enabled", "false")
 
BUCKET = "legacy-fileshare"
CATALOG = "healthcare_lakehouse"
SCHEMA = "bronze"
CHECKPOINT_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/_checkpoints/fileshare"
MIGRATION_BATCH_ID = spark.sql("SELECT uuid() AS id").collect()[0]["id"]
 
 
def _copy_one(src: str, dest: str, src_size: int) -> str:
    """Resumable: skip the copy if a file already sits at `dest` with the same byte size,
    same pattern load_legacy_fileshare.py already used for the original MinIO upload. Lets
    a killed/rerun job pick up only what's actually missing instead of redoing everything."""
    try:
        existing = dbutils.fs.ls(dest)
        if existing and existing[0].size == src_size:
            return "skipped"
    except Exception:
        pass  # doesn't exist yet at dest — fall through to copy
 
    dbutils.fs.cp(src, dest)
    return "copied"
 
 
def bulk_copy_parallel(src_path: str, dest_path: str, max_workers: int = 16) -> int:
    """Driver-side thread pool copy — validated via timed test to cut the images copy from
    an estimated 23.6 hours (serial) to ~2.6 hours (16 workers). This is plain Python
    threading, not a Spark job; it doesn't appear in the Jobs UI and runs entirely on the
    driver, same as any regular dbutils.fs.cp() call."""
    files = dbutils.fs.ls(src_path)
    tasks = [(f.path, f"{dest_path}{f.name}", f.size) for f in files if not f.isDir()]
 
    failed = []
    copied = 0
    skipped = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_copy_one, src, dst, size): src for src, dst, size in tasks}
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                if result == "skipped":
                    skipped += 1
                else:
                    copied += 1
            except Exception as e:
                failed.append((futures[future], str(e)))
 
    if failed:
        raise RuntimeError(f"{len(failed)} of {len(tasks)} files failed to copy. First few: {failed[:5]}")
 
    print(f"  copied: {copied}, already present (skipped): {skipped}")
    return len(tasks)
 
 
def ingest_fileshare_prefix(prefix: str, target_table: str, source_label: str, dest_volume: str, max_workers: int = 16):
    src_path = f"s3a://{BUCKET}/{prefix}/"
    dest_path = f"/Volumes/{CATALOG}/{SCHEMA}/{dest_volume}/{prefix}/"
    checkpoint = f"{CHECKPOINT_ROOT}/{target_table}"
 
    n_copied = bulk_copy_parallel(src_path, dest_path, max_workers=max_workers)
    print(f"[{prefix}] copied {n_copied} files into {dest_path}")
 
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "binaryFile")
        .option("cloudFiles.schemaLocation", checkpoint)
        # Earlier script iterations pointed this checkpoint at the MinIO s3a:// source
        # directly; some of that internal state (e.g. cloudFiles.source.bucket) persists
        # even after clearing the checkpoint folder. Skip option validation rather than
        # hard-failing on stale keys left over from that now-abandoned design.
        .option("cloudFiles.validateOptions", "false")
        .load(dest_path)
        .drop("content")
        .withColumn("_legacy_source_path", F.input_file_name())
        .withColumn("_legacy_source_system", F.lit(source_label))
        .withColumn("_migration_batch_id", F.lit(MIGRATION_BATCH_ID))
        .withColumn("_migrated_at", F.current_timestamp())
    )
 
    return (
        df.writeStream.format("delta")
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.{SCHEMA}.{target_table}")
    )
 
 
# --- Run all three, smallest first ------------------------------------------------------------
q_ccda = ingest_fileshare_prefix("ccda", "ccda_documents", "legacy_fileshare_ccda", "document_files")
q_ccda.awaitTermination()
 
q_fhir = ingest_fileshare_prefix("fhir", "fhir_bundles", "legacy_fileshare_fhir", "document_files")
q_fhir.awaitTermination()
 
q_images = ingest_fileshare_prefix("images", "imaging_files_catalog", "legacy_fileshare_images", "imaging_files")
q_images.awaitTermination()
 
# --- Verification: row/file counts should match confirmed source totals
# (4,596 FHIR, 4,514 C-CDA, 112,128 images — logged under Task 5) ----------------------------
for tbl in ["ccda_documents", "fhir_bundles", "imaging_files_catalog"]:
    n = spark.table(f"{CATALOG}.{SCHEMA}.{tbl}").count()
    print(f"{tbl}: {n}")
 
